# 🔍 Dataset Finder
AI-powered Kaggle dataset discovery pipeline.

**Flow:** natural-language query → LLM schema extraction → Kaggle semantic search → download → LLM validation → ranked results


## 0. Imports & setup

In [113]:
import json
import logging
import os
import re
import tempfile
from contextlib import contextmanager
from functools import lru_cache
from typing import Any

import google.generativeai as genai
import pandas as pd
import torch
from kaggle.api.kaggle_api_extended import KaggleApi
from sentence_transformers import SentenceTransformer, util
from dotenv import load_dotenv
load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
)
log = logging.getLogger(__name__)

In [114]:
print(repr(os.environ.get("GOOGLE_API_KEY")))

'AQ.Ab8RN6KpsWz4r5T4lJrYPKVbfWIdD-q9Y4Ih_GndwAv5lCPoDw'


## 1. Config

In [115]:
class Config:
    # Kaggle
    USABILITY_THRESHOLD: float = 0.5
    TOP_K_RESULTS: int = 10
    DOWNLOAD_TOP_N: int = 5          # only download the best N candidates

    # Scoring
    FIELD_MATCH_THRESHOLD: float = 0.4
    SAMPLE_VALUES_N: int = 10

    # Models
    SEMANTIC_MODEL: str = "all-MiniLM-L6-v2"
    GEMINI_MODEL: str = "gemini-3.6-flash"

    # ⚠️  Never hardcode secrets — read from environment variables
    GOOGLE_API_KEY: str = os.environ.get("GOOGLE_API_KEY", "")

## 2. Singletons
Initialised once at startup so models aren't reloaded on every call.

In [116]:
def _init_kaggle() -> KaggleApi:
    api = KaggleApi()
    api.authenticate()
    log.info("Kaggle authenticated ✅")
    return api

def _init_gemini() -> genai.GenerativeModel:
    if not Config.GOOGLE_API_KEY:
        raise EnvironmentError(
            "GOOGLE_API_KEY is not set. "
            "Run: export GOOGLE_API_KEY=<your-key>"
        )
    genai.configure(api_key=Config.GOOGLE_API_KEY)
    return genai.GenerativeModel(Config.GEMINI_MODEL)

_kaggle_api   = _init_kaggle()
_embed_model  = SentenceTransformer(Config.SEMANTIC_MODEL)
_gemini       = _init_gemini()

2026-09-12 20:06:55,178  INFO      Kaggle authenticated ✅
2026-09-12 20:06:55,182  INFO      No device provided, using cpu
2026-09-12 20:06:55,750  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-09-12 20:06:55,843  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22 "HTTP/1.1 200 OK"
2026-09-12 20:06:56,200  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-09-12 20:06:56,294  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f

## 3. Step 1 — Query extraction
Parse a natural-language query into a structured dataset spec via Gemini.

In [117]:
_EXTRACTION_PROMPT = """
You are an AI system for extracting dataset requirements for data analysis.

Responsibilities:
1. Understand the dataset being requested, independent of the user's analytical goal.
2. Identify the data context: timeseries, geographic, cross-sectional, panel,
   list-based, event-log, or hierarchical.
3. Use the user's goal only to infer fields — do NOT include it in topic/sub_topic.
4. Extract:
   - topic          → short phrase summarising the dataset
   - sub_topic      → main subject category (1-2 words)
   - context        → one of the data-context categories above
   - desired_fields → fields explicitly or implicitly requested; infer if missing
   - extra_fields   → additional helpful fields beyond desired_fields

Return ONLY valid JSON. No markdown, no explanation.

### Examples

User: "Give me population data for China from past to present to predict future growth."
Output:
{{
  "topic": "Historical population of China",
  "sub_topic": "population",
  "context": "timeseries",
  "desired_fields": ["year", "population"],
  "extra_fields": ["province", "gdp", "fertility_rate"]
}}

User: "{query}"
"""

_REQUIRED_KEYS = {"topic", "sub_topic", "context", "desired_fields"}

def extract_query(query: str) -> dict[str, Any]:
    prompt = _EXTRACTION_PROMPT.format(query=query)
    response = _gemini.generate_content(prompt)
    text = response.text.replace("```json", "").replace("```", "").strip()

    try:
        result = json.loads(text)
    except json.JSONDecodeError as exc:
        raise ValueError(f"LLM returned invalid JSON:\n{text}") from exc

    missing = _REQUIRED_KEYS - result.keys()
    if missing:
        raise ValueError(f"LLM response missing keys: {missing}")

    result.setdefault("extra_fields", [])
    log.info("Extracted → topic: %r | context: %r", result["topic"], result["context"])
    return result

## 4. Step 2 — Kaggle search + semantic re-ranking
Search Kaggle, filter by usability rating, then re-rank results with MiniLM cosine similarity.

In [118]:
@lru_cache(maxsize=Config.TOP_K_RESULTS)
def search_kaggle(topic: str) -> list[str]:
    try:
        candidates = [
            d for d in _kaggle_api.dataset_list(search=topic)
            if (d.usability_rating or 0) >= Config.USABILITY_THRESHOLD
        ]
    except Exception as exc:
        log.error("Kaggle search failed: %s", exc)
        return []

    if not candidates:
        log.warning("No Kaggle results for %r", topic)
        return []

    titles     = [d.title or "" for d in candidates]
    subtitles  = [d.subtitle or d.title or "" for d in candidates]

    topic_emb      = _embed_model.encode(topic,      convert_to_tensor=True)
    titles_emb     = _embed_model.encode(titles,     convert_to_tensor=True)
    subtitles_emb  = _embed_model.encode(subtitles,  convert_to_tensor=True)

    scores = (
        util.cos_sim(topic_emb, titles_emb)[0]
        + util.cos_sim(topic_emb, subtitles_emb)[0]
    ) / 2

    k = min(Config.TOP_K_RESULTS, len(scores))
    top_indices = torch.topk(scores, k=k).indices.tolist()
    refs = [candidates[i].ref for i in top_indices]

    log.info("Found %d datasets for %r", len(refs), topic)
    return refs

## 5. Step 3 — Download datasets
Context manager: only downloads `DOWNLOAD_TOP_N` candidates.
Temp directory is guaranteed to be cleaned up even if an error occurs.

In [119]:
@contextmanager
def download_datasets(refs: list[str], top_n: int = Config.DOWNLOAD_TOP_N):
    """
    Usage:
        with download_datasets(refs) as folder:
            dfs = load_csvs(folder)
    """
    tmp = tempfile.TemporaryDirectory()
    try:
        for ref in refs[:top_n]:
            log.info("Downloading %s …", ref)
            try:
                _kaggle_api.dataset_download_files(ref, path=tmp.name, unzip=True)
            except Exception as exc:
                log.warning("Skipping %s: %s", ref, exc)
        yield tmp.name
    finally:
        tmp.cleanup()
        log.info("Temp directory cleaned up.")

## 6. Step 4 — Load CSVs

In [120]:
def load_csvs(folder: str) -> dict[str, pd.DataFrame]:
    dfs: dict[str, pd.DataFrame] = {}
    for root, _, files in os.walk(folder):
        for fname in files:
            if not fname.endswith(".csv"):
                continue
            path = os.path.join(root, fname)
            try:
                dfs[fname] = pd.read_csv(path, encoding_errors="replace")
            except Exception as exc:
                log.warning("Could not read %s: %s", fname, exc)
    log.info("Loaded %d CSV files.", len(dfs))
    return dfs

## 7. Step 5 — Validation
Two-stage scoring:
1. **Keyword score** — fast column-name substring matching (desired + extra fields weighted separately)
2. **LLM score** — Gemini inspects sample values and scores semantic relevance independently

In [121]:
def _normalize(s: str) -> str:
    return re.sub(r"[\s_\-]+", "", s.lower())

def _field_match_score(
    df: pd.DataFrame,
    desired: list[str],
    extra: list[str],
) -> tuple[float, float]:
    """Returns (desired_score, extra_score) in [0, 1] — scored separately."""
    col_norms = {_normalize(c) for c in df.columns}

    def _hits(fields: list[str]) -> int:
        return sum(
            any(_normalize(f) in cn for cn in col_norms)
            for f in fields
        )

    d_score = _hits(desired) / len(desired) if desired else 0.0
    e_score = _hits(extra)   / len(extra)   if extra   else 0.0
    return min(d_score, 1.0), min(e_score, 1.0)

def _sample_values(df: pd.DataFrame, n: int = Config.SAMPLE_VALUES_N) -> str:
    lines = []
    for col in df.columns:
        unique = df[col].dropna().unique()[:n].tolist()
        lines.append(f"  {col}: {unique}")
    return "\n".join(lines)

_VALIDATION_PROMPT = """
You are validating whether a CSV dataset matches a user's data request.

User request:
  Topic    : {topic}
  Sub-topic: {sub_topic}
  Context  : {context}
  Original query: "{query}"

Dataset: {name}
Columns: {columns}
Keyword match score (desired fields): {keyword_score:.0%}

Sample values (up to {n} unique per column):
{sample_values}

Tasks:
1. Assess whether the VALUES match the query context.
2. Produce an independent relevance_score in [0.0, 1.0].
3. One-sentence reasoning citing specific observed values.
4. Concrete use_case inferred from actual columns/values (do NOT paraphrase the query).

Return ONLY valid JSON, no markdown:
{{
  "relevance_score": 0.85,
  "reasoning": "...",
  "use_case": "..."
}}
"""

def _llm_validate(
    name: str,
    df: pd.DataFrame,
    query_info: dict,
    query: str,
    keyword_score: float,
) -> dict[str, Any] | None:
    prompt = _VALIDATION_PROMPT.format(
        topic=query_info["topic"],
        sub_topic=query_info["sub_topic"],
        context=query_info["context"],
        query=query,
        name=name,
        columns=list(df.columns),
        keyword_score=keyword_score,
        n=Config.SAMPLE_VALUES_N,
        sample_values=_sample_values(df),
    )
    try:
        response = _gemini.generate_content(prompt)
        text = response.text.replace("```json", "").replace("```", "").strip()
        return json.loads(text)
    except Exception as exc:
        log.warning("LLM validation failed for %s: %s", name, exc)
        return None

def validate_datasets(
    dataframes: dict[str, pd.DataFrame],
    query_info: dict,
    query: str,
) -> list[dict[str, Any]]:
    desired = query_info["desired_fields"]
    extra   = query_info.get("extra_fields", [])
    results = []

    for name, df in dataframes.items():
        d_score, e_score = _field_match_score(df, desired, extra)
        combined_keyword = 0.7 * d_score + 0.3 * e_score   # desired weighted higher

        if combined_keyword < Config.FIELD_MATCH_THRESHOLD:
            log.debug("Pre-filter skip %s (keyword=%.0%%)", name, combined_keyword)
            continue

        log.info("Validating %s (keyword=%.0%%)", name, combined_keyword)
        llm = _llm_validate(name, df, query_info, query, combined_keyword)
        if llm is None:
            continue

        results.append({
            "name":            name,
            "keyword_score":   round(combined_keyword, 3),
            "relevance_score": round(llm.get("relevance_score", 0.0), 3),
            "reasoning":       llm.get("reasoning", ""),
            "use_case":        llm.get("use_case", ""),
            "dataframe":       df,
        })

    results.sort(key=lambda r: r["relevance_score"], reverse=True)
    log.info("Validated %d datasets.", len(results))
    return results

## 8. Pipeline orchestrator
Single entry point — call `find_datasets(query)` and get back a ranked list.

In [122]:
def find_datasets(query: str) -> list[dict[str, Any]]:
    """
    Full pipeline: natural-language query → ranked list of matching datasets.

    Each result dict:
        name, keyword_score, relevance_score, reasoning, use_case, dataframe
    """
    query_info  = extract_query(query)
    search_term = f"{query_info['topic']} {query_info['sub_topic']}"
    refs        = search_kaggle(search_term)

    if not refs:
        log.warning("No Kaggle datasets found.")
        return []

    with download_datasets(refs, top_n=Config.DOWNLOAD_TOP_N) as folder:
        dataframes = load_csvs(folder)
        if not dataframes:
            log.warning("No CSV files found in downloaded datasets.")
            return []
        results = validate_datasets(dataframes, query_info, query)

    return results

## 9. Run it

In [123]:
# %%time
query = "I need a dataset of population in China by province"
results = find_datasets(query)

if not results:
    print("No matching datasets found.")
else:
    for rank, d in enumerate(results, 1):
        print(f"\n#{rank}  {d['name']}")
        print(f"    Relevance : {d['relevance_score']:.0%}  |  Keyword: {d['keyword_score']:.0%}")
        print(f"    Reasoning : {d['reasoning']}")
        print(f"    Use case  : {d['use_case']}")
        print(f"    Shape     : {d['dataframe'].shape}")

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash
Please retry in 4.725568699s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-3.6-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 4
}
]

In [ ]:
%%time
query2 = "I need an employees dataset"
results2 = find_datasets(query2)

if not results:
    print("No matching datasets found.")
else:
    for rank, d in enumerate(results2, 1):
        print(f"\n#{rank}  {d['name']}")
        print(f"    Relevance : {d['relevance_score']:.0%}  |  Keyword: {d['keyword_score']:.0%}")
        print(f"    Reasoning : {d['reasoning']}")
        print(f"    Use case  : {d['use_case']}")
        print(f"    Shape     : {d['dataframe'].shape}")

2026-09-12 20:03:55,936  INFO      Extracted → topic: 'Employee information' | context: 'cross-sectional'
Batches: 100%|██████████| 1/1 [00:00<00:00, 15.50it/s]
2026-09-12 20:03:57,050  INFO      Found 10 datasets for 'Employee information human resources'
2026-09-12 20:03:57,052  INFO      Downloading davidepolizzi/hr-data-set-based-on-human-resources-data-set …


Dataset URL: https://www.kaggle.com/datasets/davidepolizzi/hr-data-set-based-on-human-resources-data-set


2026-09-12 20:03:59,484  INFO      Downloading umairziact/employees-data …


Dataset URL: https://www.kaggle.com/datasets/umairziact/employees-data


2026-09-12 20:04:01,963  INFO      Downloading ravindrasinghrana/employeedataset …


...

Dataset URL: https://www.kaggle.com/datasets/saadharoon27/hr-analytics-dataset


2026-09-12 20:04:06,774  INFO      Downloading mrhainguyen/employee-names-salaries-and-position-titles …


Dataset URL: https://www.kaggle.com/datasets/mrhainguyen/employee-names-salaries-and-position-titles


2026-09-12 20:04:09,690  INFO      Loaded 10 CSV files.
--- Logging error ---
Traceback (most recent call last):
  File "c:\Users\PC\AppData\Local\Python\pythoncore-3.14-64\Lib\logging\__init__.py", line 1151, in emit
    msg = self.format(record)
  File "c:\Users\PC\AppData\Local\Python\pythoncore-3.14-64\Lib\logging\__init__.py", line 999, in format
    return fmt.format(record)
           ~~~~~~~~~~^^^^^^^^
  File "c:\Users\PC\AppData\Local\Python\pythoncore-3.14-64\Lib\logging\__init__.py", line 712, in format
    record.message = record.getMessage()
                     ~~~~~~~~~~~~~~~~~^^
  File "c:\Users\PC\AppData\Local\Python\pythoncore-3.14-64\Lib\logging\__init__.py", line 400, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
ValueError: unsupported format character '%' (0x25) at index 26
Call stack:
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\PC\AppData\Roaming\Python\Python314\si

CPU times: total: 5.77 s
Wall time: 19 s


## 10. Inspect top result

In [ ]:
if results:
    best = results[0]
    print(f"Dataset: {best['name']}")
    display(best['dataframe'].head(10))

Dataset: Philippine_Volcanoes.csv


,Volcano Name,Latitude,Longitude,Province,State,Is Cone Field,Comment From BwandoWando
0,Abongabong,6.503000,121.984280,Basilan,Inactive,No,NaN
1,Abunug,11.012000,124.690500,Southern Leyte,Inactive,No,NaN
2,Abuyug,10.794919,124.964104,Leyte,Inactive,No,NaN
3,Aguada,10.833333,121.033333,Palawan,Inactive,No,NaN
4,Agutaya,11.150000,120.950000,Palawan,Inactive,No,NaN
5,Akir-akir,7.251968,124.548160,Maguindanao,Inactive,No,NaN
6,Alligator,14.182436,121.206106,Laguna,Inactive,No,NaN
7,Alto,11.108333,124.741700,Leyte,Inactive,No,NaN
8,Alu,5.687940,120.881403,Sulu,Inactive,No,NaN
9,Ambalatungan Group,17.311620,121.097310,Kalinga,Inactive,No,NaN


In [ ]:
best2 = results2[0]
print(f"Dataset: {best2['name']}")
display(best2['dataframe'].head(10))

Dataset: employee_master.csv


,Employee ID,Employee Name,Department,Designation,Joining Date,Employment Type,Basic Salary,Payment Channel,Account Reference,Tax Status,PF Eligible,Employment Status
0,NPS-1001,Aarav Hasan,Human Resources,HR Executive,44576,Permanent,42000,Bank Transfer,•••• 4300,Below Threshold,Yes,Active
1,NPS-1002,Nadia Rahman,Human Resources,Senior HR Officer,44409,Permanent,56000,Bank Transfer,•••• 4317,Taxable,Yes,Active
2,NPS-1003,Tanvir Hossain,Finance,Accounts Executive,44997,Permanent,48000,Bank Transfer,•••• 4334,Taxable,Yes,Active
3,NPS-1004,Maliha Noor,Finance,Finance Analyst,44141,Permanent,62000,Bank Transfer,•••• 4351,Taxable,Yes,Active
4,NPS-1005,Rafi Ahmed,Sales,Sales Executive,45332,Permanent,35000,Bank Transfer,•••• 4368,Below Threshold,Yes,Active
5,NPS-1006,Farzana Karim,Sales,Key Account Officer,44822,Permanent,45000,Bank Transfer,•••• 4385,Taxable,Yes,Active
6,NPS-1007,Samin Chowdhury,Information Technology,Software Engineer,44354,Permanent,72000,Bank Transfer,•••• 4402,Taxable,Yes,Active
7,NPS-1008,Tanjila Islam,Information Technology,IT Support Officer,45662,Contract,38000,Mobile Financial Service,•••• 4419,Below Threshold,No,Active
8,NPS-1009,Mahin Kabir,Operations,Operations Executive,45067,Permanent,41000,Bank Transfer,•••• 4436,Below Threshold,Yes,Active
9,NPS-1010,Ayesha Sultana,Operations,Operations Coordinator,44896,Permanent,46000,Bank Transfer,•••• 4453,Taxable,Yes,Active


In [ ]:
len(best2['dataframe'])

24